# Stock Price Prediction Using Machine Learning
### Time-Series Forecasting Project | TCS (Tata Consultancy Services)

**Project objective:** Use historical stock-market data to estimate the next trading session's closing price.

This project follows a simple supervised machine-learning workflow, but uses a time-series dataset and time-aware evaluation.

**Dataset:** Daily TCS Ltd. stock data (TCS.NS) downloaded from Yahoo Finance using yfinance.

**Models:** Linear Regression and Random Forest Regressor, compared with a previous-close baseline.

> **Important:** Educational demonstration only. Stock prices are uncertain; predictions are not investment advice and must not be used as a basis for trading.

## 1. Import required libraries
Run this cell first. If a package is missing, install the packages listed in requirements.txt.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

## 2. Download the dataset
The end date is exclusive. If Yahoo Finance is temporarily unavailable, rerun this cell later.

In [ ]:
TICKER = "TCS.NS"
START_DATE = "2019-01-01"
END_DATE = None  # None means the latest available data

data = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)

# Recent yfinance versions may return MultiIndex columns for a single ticker.
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

if data.empty:
    raise ValueError("No data was downloaded. Check your internet connection or ticker symbol.")

data = data.reset_index()
data.columns = [str(col).strip().replace(" ", "_") for col in data.columns]

print("Rows downloaded:", len(data))
print("Available columns:", list(data.columns))
data.head()

## 3. Understand and clean the data
We use the adjusted OHLCV series returned by auto_adjust=True.

In [ ]:
required_columns = ["Date", "Close", "Volume"]
missing = [col for col in required_columns if col not in data.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

data["Date"] = pd.to_datetime(data["Date"])
data = data.sort_values("Date").drop_duplicates(subset="Date")
data["Close"] = pd.to_numeric(data["Close"], errors="coerce")
data["Volume"] = pd.to_numeric(data["Volume"], errors="coerce")
data = data.dropna(subset=["Close", "Volume"]).reset_index(drop=True)

print("Date range:", data["Date"].min().date(), "to", data["Date"].max().date())
print("Clean rows:", len(data))
data[["Date", "Close", "Volume"]].tail()

## 4. Explore the stock-price trend
Historical trends do not guarantee future performance.

In [ ]:
plt.plot(data["Date"], data["Close"], label="TCS closing price")
plt.title("TCS Historical Closing Price")
plt.xlabel("Date")
plt.ylabel("Price (INR)")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Feature engineering
Lag and rolling features use information available up to the current trading day. The target is the next observed trading session's closing price.

In [ ]:
model_data = data[["Date", "Close", "Volume"]].copy()

model_data["Return_1D"] = model_data["Close"].pct_change()
model_data["Close_Lag_1"] = model_data["Close"]
model_data["Close_Lag_2"] = model_data["Close"].shift(1)
model_data["Close_Lag_3"] = model_data["Close"].shift(2)
model_data["MA_5"] = model_data["Close"].rolling(window=5).mean()
model_data["MA_10"] = model_data["Close"].rolling(window=10).mean()
model_data["Volume_Lag_1"] = model_data["Volume"]
model_data["Next_Close"] = model_data["Close"].shift(-1)

feature_columns = ["Close_Lag_1", "Close_Lag_2", "Close_Lag_3", "MA_5", "MA_10", "Return_1D", "Volume_Lag_1"]
latest_features = model_data[feature_columns].tail(1).copy()
supervised = model_data.dropna(subset=feature_columns + ["Next_Close"]).copy()

print("Feature columns:", feature_columns)
print("Rows available for modelling:", len(supervised))
supervised[["Date"] + feature_columns + ["Next_Close"]].head()

## 6. Split the data chronologically
Train on the older 80% and test on the newest 20%; rows are not randomly shuffled.

In [ ]:
split_index = int(len(supervised) * 0.80)
train = supervised.iloc[:split_index].copy()
test = supervised.iloc[split_index:].copy()

X_train = train[feature_columns]
y_train = train["Next_Close"]
X_test = test[feature_columns]
y_test = test["Next_Close"]

print("Training observations:", len(train))
print("Testing observations:", len(test))
print("Training period:", train["Date"].min().date(), "to", train["Date"].max().date())
print("Testing period:", test["Date"].min().date(), "to", test["Date"].max().date())

## 7. Train the machine-learning models
Linear Regression learns a linear relationship. Random Forest combines decision trees to learn non-linear patterns.

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_predictions = linear_model.predict(X_test)

forest_model = RandomForestRegressor(n_estimators=250, max_depth=8, min_samples_leaf=3, random_state=42, n_jobs=-1)
forest_model.fit(X_train, y_train)
forest_predictions = forest_model.predict(X_test)

# Baseline: next close equals current close.
baseline_predictions = test["Close_Lag_1"].to_numpy()

## 8. Evaluate model performance
Lower MAE and RMSE generally indicate smaller errors. R-squared may be negative. Compare ML models with the previous-close baseline.

In [ ]:
def regression_metrics(actual, predicted, model_name):
    return {"Model": model_name, "MAE": mean_absolute_error(actual, predicted), "RMSE": np.sqrt(mean_squared_error(actual, predicted)), "R2": r2_score(actual, predicted)}

results = pd.DataFrame([
    regression_metrics(y_test, baseline_predictions, "Previous Close Baseline"),
    regression_metrics(y_test, linear_predictions, "Linear Regression"),
    regression_metrics(y_test, forest_predictions, "Random Forest")
]).sort_values("RMSE").reset_index(drop=True)

results.round(3)

## 9. Actual vs predicted prices
Compare observed next-session closing prices with estimates over the held-out test period.

In [ ]:
plt.plot(test["Date"], y_test.to_numpy(), label="Actual next close", linewidth=2)
plt.plot(test["Date"], linear_predictions, label="Linear Regression", alpha=0.85)
plt.plot(test["Date"], forest_predictions, label="Random Forest", alpha=0.85)
plt.title("Actual vs Predicted TCS Closing Prices (Test Set)")
plt.xlabel("Date")
plt.ylabel("Price (INR)")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Feature importance (Random Forest)
Importance is a rough measure of model reliance, not proof of causation.

In [ ]:
importance = pd.Series(forest_model.feature_importances_, index=feature_columns).sort_values()
importance.plot(kind="barh")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 11. Experimental next-session estimate
Refit on all labelled rows and estimate the next observed session. This is experimental, not a reliable forecast.

In [ ]:
final_linear = LinearRegression()
final_linear.fit(supervised[feature_columns], supervised["Next_Close"])
next_linear_estimate = float(final_linear.predict(latest_features)[0])

final_forest = RandomForestRegressor(n_estimators=250, max_depth=8, min_samples_leaf=3, random_state=42, n_jobs=-1)
final_forest.fit(supervised[feature_columns], supervised["Next_Close"])
next_forest_estimate = float(final_forest.predict(latest_features)[0])

print("Latest available trading date:", data["Date"].iloc[-1].date())
print("Latest closing price (INR):", round(float(data["Close"].iloc[-1]), 2))
print("Linear Regression next-session estimate (INR):", round(next_linear_estimate, 2))
print("Random Forest next-session estimate (INR):", round(next_forest_estimate, 2))

## 12. Conclusion
We collected TCS data, engineered lag and rolling features, used a chronological split, trained two regression models, and compared them with a previous-close baseline. The lowest test RMSE identifies the best result for this test period only. If a model does not beat the baseline, that is a valid finding.

### Future improvements
- Add other stock tickers.
- Use walk-forward validation.
- Explore ARIMA or LSTM methods.
- Include broader market variables.

### Data source
Yahoo Finance historical market data via yfinance. Data is downloaded at runtime; no static CSV is bundled.